In [5]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from PIL import Image

In [14]:
DIR = "/mnt/c/users/helen/Desktop/Ayriscan/WT/foci"
image_path = "/mnt/c/users/helen/Desktop/Ayriscan/WT/c1_wt-01-airyscan_processing-29_roi_0838-1294_bg_subtracted.tif"
df_path = "/mnt/c/users/helen/Desktop/Ayriscan/WT/foci/c1_wt-01-airyscan_processing-29_roi_0838-1294_bg_subtracted_foci.csv"


In [15]:
df = pd.read_csv(df_path)

In [16]:
df

,id,x [nm],y [nm],sigma [nm],intensity [photon]
0,1.0,40563.370234,33360.711731,43.287275,5256.691286
1,2.0,40584.371917,33689.483454,50.451307,7769.935204
2,3.0,40660.768577,31657.778825,41.159164,2660.325640
3,4.0,40679.458481,34227.352053,50.044643,3594.164336
4,5.0,40725.776546,31442.036060,51.013373,5632.184483
...,...,...,...,...,...
898,899.0,50716.543772,26772.254771,50.653734,9790.454703
899,900.0,50714.795240,25719.673122,49.495487,8510.300403
900,901.0,50757.528262,27110.051554,31.498827,1452.150330
901,902.0,50790.501769,25998.336508,51.731932,9785.518396


In [13]:
image = Image.open(image_path)  # load image
arr = np.array(image) # convert image to numpy matrix
H, W = arr.shape # get number of pixels (512*512 for 16-bit image)

# Storage lists
x_list = []
y_list = []
sigma_list = []
mean_list = []


In [ ]:
df.columns = df.columns.str.strip()  # remove hidden spaces in headers
df = df.rename(columns={"x [nm]": "x_nm",
                            "y [nm]": "y_nm",
                            "sigma [nm]": "sigma_nm",
                            "intensity [photon]": "intensity_photon"}) # Rename columns
# Iteration through the thunderSTORM dataframe
for _, row in df.iterrows():
    x_px = int(row["x_nm"] / px_size_nm)
    y_px = int(row["y_nm"] / px_size_nm)
    r_px = max(1, int(row["sigma_nm"] / px_size_nm)) # minimal possible value for radius is 1 pixel!

    # Build circular mask (clipped automatically)
    rr, cc = disk((y_px, x_px), r_px, shape=(H, W))
    mask = np.zeros((H, W), dtype=bool)
    mask[rr, cc] = True

    n_pixels_mask = np.sum(mask)

    # Compute mean intensity
        if n_pixels_mask > 0:
            mean_intensity = arr[mask].mean()
        else:
            mean_intensity = np.nan

        # Add values to the corresponding lists
        x_list.append(x_px)
        y_list.append(y_px)
        sigma_list.append(r_px)
        mean_list.append(mean_intensity)
    
    # Return modified copy
    df_out = df.copy()
    df_out["x_pixel"] = x_list
    df_out["y_pixel"] = y_list
    df_out["sigma_pixel"] = sigma_list
    df_out["foci_MFI"] = mean_list

In [20]:
df

,id,x_nm,y_nm,sigma_nm,intensity_photon
0,1.0,40563.370234,33360.711731,43.287275,5256.691286
1,2.0,40584.371917,33689.483454,50.451307,7769.935204
2,3.0,40660.768577,31657.778825,41.159164,2660.325640
3,4.0,40679.458481,34227.352053,50.044643,3594.164336
4,5.0,40725.776546,31442.036060,51.013373,5632.184483
...,...,...,...,...,...
898,899.0,50716.543772,26772.254771,50.653734,9790.454703
899,900.0,50714.795240,25719.673122,49.495487,8510.300403
900,901.0,50757.528262,27110.051554,31.498827,1452.150330
901,902.0,50790.501769,25998.336508,51.731932,9785.518396
